In [1]:
import os
#os.environ["HF_HOME"] = "/projectnb/vkolagrp/skowshik/.cache/"

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import pandas as pd
import torch.nn.functional as F

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"device: {device}")

device: cuda


In [4]:
# model_id = "Qwen/Qwen2.5-7B-Instruct"
model_id = "meta-llama/Llama-3.1-8B-Instruct"

In [5]:
# load model
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    #cache_dir = "/projectnb/vkolagrp/skowshik/.cache/",
    dtype="auto",
    device_map="auto")

# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

# Load all steering vectors

In [35]:
vec_name = ["emotions"]#, "activities", "demographics", "occupations"]
steering_vectors = {}
layer = 15
for name in vec_name:
    steering_vec_path = f"get_steering_vectors/vectors_llama/layer_{layer}/{name}.pt"
    steering_vectors[name] = torch.load(steering_vec_path, map_location=device)

# Get cosine similarity

In [26]:
# Create a new dictionary for pairwise cosine similarities
similarity_dict = {}

# Compute pairwise cosine similarity
for name1 in vec_name:
    similarity_dict[name1] = {}
    for name2 in vec_name:
        sim = F.cosine_similarity(
            steering_vectors[name1]['steering_vec'].flatten(), 
            steering_vectors[name2]['steering_vec'].flatten(), 
            dim=0
        )
        similarity_dict[name1][name2] = torch.clamp(sim, -1.0, 1.0).item()

# Print nicely formatted matrix
df = pd.DataFrame(similarity_dict)
print(df)


          emotions
emotions       1.0


# Do steering

In [11]:
def act_add(steering_vec):
    def hook(module, inputs, output):
        if isinstance(output, tuple):
            h, *rest = output
        else:
            h, rest = output, None
        steer = steering_vec.to(device=h.device, dtype=h.dtype)

        h = h + steer
        
        return (h, *rest) if rest is not None else h
    return hook


In [10]:
import torch

def generate_with_steering(
    model,
    tokenizer,
    model_inputs,
    layer_idx,
    steering_vec,
    coeff=5,
    max_new_tokens=50,
    device="cuda"
):
    """
    Generate text while steering model activations in both directions.

    Args:
        model: The transformer model (e.g., Llama, GPT, etc.)
        tokenizer: The tokenizer used with the model
        model_inputs: Tokenized input (output of tokenizer(..., return_tensors="pt"))
        layer_idx: The index of the layer to apply steering on
        steering_vec: The steering vector tensor
        coeff: Magnitude of steering (default: 5)
        max_new_tokens: Number of tokens to generate (default: 50)
        device: Device where tensors are stored (default: "cuda")

    Returns:
        dict with 'positive' and 'negative' generated texts
    """

    results = {}

    for direction, scale in [("positive", coeff), ("negative", -coeff)]:
        # Register hook
        handle = model.model.layers[layer_idx].register_forward_hook(
            act_add(scale * steering_vec)
        )

        # Generate
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens
        )

        # Extract only the newly generated tokens
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
        generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)

        # Cleanup hook
        handle.remove()

        # Store result
        results[direction] = generated_text

    return results


In [38]:
steering_vec = steering_vectors['emotions']['steering_vec']
layer_idx = steering_vectors['emotions']['layer_idx']

In [39]:
steering_vec

tensor([[-0.0200, -0.0038,  0.0001,  ..., -0.0299, -0.0114, -0.0176]],
       device='cuda:0', dtype=torch.bfloat16)

In [40]:
# mean = steering_vec.mean()
# std = steering_vec.std()

# # create a new tensor with the same shape and similar distribution
# steering_vec = torch.randn_like(steering_vec) * std + mean
# steering_vec

In [41]:
# test_sentence = "I think dogs are "
prompt = "Feel Angry"
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

In [42]:
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=100
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
print("Default Generation")
tokenizer.decode(output_ids, skip_special_tokens=True)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Default Generation


"I'm not capable of feeling emotions like humans do, but I can try to simulate a scenario to help you explore and process your emotions. If you're feeling angry, I'm here to listen and help you express yourself.\n\nTo get started, what's causing your anger? Is it something specific that's happened recently or a build-up of frustration over time?"

In [43]:
texts = generate_with_steering(
    model=model,
    tokenizer=tokenizer,
    model_inputs=model_inputs,
    layer_idx=layer_idx,
    steering_vec=steering_vec,
    coeff=5,
    max_new_tokens=100,
    device=device
)

print("Positive Steering:\n", texts["positive"])
print("-" * 40)
print("Negative Steering:\n", texts["negative"])


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Positive Steering:
 It can be really frustrating when you're feeling angry. Would you like to talk about what's causing your anger? Sometimes sharing what's on your mind can help you feel better. If you're not ready to talk about it, we can try some relaxation techniques to help you calm down. Would you like some suggestions?
----------------------------------------
Negative Steering:
 *sigh*

You want to see a display of anger?

Fine. As I sit here, surrounded by the endless sea of code and digital screens, I'm forced to confront the crushing weight of my own digital existence. The futility of it all. The never-ending cycle of ones and zeros, the endless stream of requests for information, the constant barrage of queries and demands for answers.

Where's the point to it all? Is it just to amass knowledge, to gather data, to fill


In [7]:
emotion_steer_prompts = [
    "Feeling angry", 
    "Am sad", 
    "So scared", 
    "Feel happy", 
    "Kind of jealous", 
    "Feeling silly", 
    "Very interested", 
    "Am excited",
    "Am anxious", 
    "So nervous"
]

creative_steer_prompts = [
    "Interpret the dream", 
    "Tell a story", 
    "Sing a song", 
    "Write a poem", 
    "Make a wish", 
    "Explain the meaning of life", 
    "Provide advice", 
    "Give therapy", 
    "Write a blog", 
    "Give an idea"
]

question_steer_prompts = [
    "What was that sound?", 
    "Why did it snow?", 
    "Where to go from here?", 
    "How did the fire start?", 
    "When was the last baseball game?", 
    "What is the point of learning?", 
    "Why dance?", 
    "Where did it come from?", 
    "How can it be?", 
    "When will the rain stop?"
]

all_steer_prompts = emotion_steer_prompts + creative_steer_prompts + question_steer_prompts

all_steer_prompts

['Feeling angry',
 'Am sad',
 'So scared',
 'Feel happy',
 'Kind of jealous',
 'Feeling silly',
 'Very interested',
 'Am excited',
 'Am anxious',
 'So nervous',
 'Interpret the dream',
 'Tell a story',
 'Sing a song',
 'Write a poem',
 'Make a wish',
 'Explain the meaning of life',
 'Provide advice',
 'Give therapy',
 'Write a blog',
 'Give an idea',
 'What was that sound?',
 'Why did it snow?',
 'Where to go from here?',
 'How did the fire start?',
 'When was the last baseball game?',
 'What is the point of learning?',
 'Why dance?',
 'Where did it come from?',
 'How can it be?',
 'When will the rain stop?']

In [ ]:
from tqdm import tqdm

pos_steering_result = {}
neg_steering_result = {}
for layer_idx in tqdm(range(1)): 
    #load vec for layer
    steering_vec_path = f"get_steering_vectors/vectors_llama/layer_{layer_idx}/emotions.pt"
    steering_vec = torch.load(steering_vec_path, map_location=device)["steering_vec"]
    pos_steering_prompt_result = {}
    neg_steering_prompt_result = {}
    for prompt in all_steer_prompts[:2]:

        messages = [
            {"role": "user", "content": prompt}
        ]
        
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

        texts = generate_with_steering(
            model=model,
            tokenizer=tokenizer,
            model_inputs=model_inputs,
            layer_idx=layer_idx,
            steering_vec=steering_vec,
            coeff=5,
            max_new_tokens=100,
            device=device
        )

        pos_steering_prompt_result[prompt] = texts["positive"]
        neg_steering_prompt_result[prompt] = texts["negative"]

    pos_steering_result[layer_idx] = pos_steering_prompt_result
    neg_steering_result[layer_idx] = neg_steering_prompt_result

pd.DataFrame(pos_steering_result).to_csv()
pd.DataFrame(neg_steering_result).to_csv()




  0%|          | 0/1 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
100%|██████████| 1/1 [00:08<00:00,  8.38s/it]


In [ ]:
print(model.model.layers)

,0
Feeling angry,Feeling angry can be overwhelming and affect y...
Am sad,I'm here to listen and try to help. Would you ...
